In [ ]:
pip install scipy

In [ ]:
import scipy
print(scipy.__version__)

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog

# ==========================================
# PARAMETERS
# ==========================================

AVAILABLE_HOURS = 22

book_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_13march_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

# ==========================================
# LOAD DATA
# ==========================================

hz_parts = pd.read_excel(book_path, sheet_name="HZ")
vt_parts = pd.read_excel(book_path, sheet_name="VT")

stats = pd.read_excel(book_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")

data = stats.merge(daily, left_on="Part", right_on="Material")

# ==========================================
# PRODUCTION RATE
# ==========================================

data["Rate"] = (3600 / data["Cycle time "].replace(0, np.nan)) * data["cavity"]

data = data[data["Rate"].notna()]

# ==========================================
# PARTS AND MACHINES
# ==========================================

parts = data["Material"].unique().tolist()

machines = hz_matrix.columns[1:].tolist()

# ==========================================
# LOOKUP DICTIONARIES
# ==========================================

inventory = dict(zip(data["Material"], data["Inventory on 24th"]))

demand = dict(zip(data["Material"], data["2026-03-12 Total Production Plan"]))

rate = dict(zip(data["Material"], data["Rate"]))

# ==========================================
# COMPATIBILITY
# ==========================================

compatibility = {}

for _, row in hz_matrix.iterrows():

    part = row["Part"]

    for m in machines:

        compatibility[(part, m)] = row[m]

# ==========================================
# CREATE VARIABLES
# ==========================================

variables = []
var_index = {}

k = 0

for p in parts:
    for m in machines:

        if compatibility.get((p, m), 0) == 1:

            variables.append((p, m))
            var_index[(p, m)] = k
            k += 1

n = len(variables)

# ==========================================
# OBJECTIVE
# ==========================================

c = np.ones(n) * 1

# ==========================================
# CONSTRAINT MATRICES
# ==========================================

A = []
b = []

# ------------------------------------------
# MACHINE CAPACITY
# ------------------------------------------

for m in machines:

    row = np.zeros(n)

    for p in parts:

        if (p, m) in var_index:

            idx = var_index[(p, m)]
            row[idx] = 1 / rate[p]

    A.append(row)
    b.append(AVAILABLE_HOURS)

# ------------------------------------------
# DEMAND CONSTRAINT
# ------------------------------------------

for p in parts:

    row = np.zeros(n)

    for m in machines:

        if (p, m) in var_index:

            idx = var_index[(p, m)]
            row[idx] = -1

    A.append(row)
    b.append(inventory.get(p, 0) - demand.get(p, 0))

# ==========================================
# SOLVE OPTIMIZATION
# ==========================================

result = linprog(
    c,
    A_ub=np.array(A),
    b_ub=np.array(b),
    bounds=(0, None),
    method="highs"
)

# ==========================================
# EXTRACT PLAN
# ==========================================

solution = result.x

plan = []

for (p, m), idx in var_index.items():

    qty = solution[idx]

    if qty > 1:

        plan.append({
            "Part": p,
            "Machine": m,
            "Qty": round(qty, 0),
            "Run_Hours": round(qty / rate[p], 2)
        })

plan_df = pd.DataFrame(plan)

plan_df.to_excel("Smart_APS_Plan_V1.xlsx", index=False)

print("Optimization Completed")

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog

# ==========================================
# PARAMETERS
# ==========================================

AVAILABLE_HOURS = 22
SHORTAGE_PENALTY = 100

book_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_13march_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

# ==========================================
# LOAD DATA
# ==========================================

hz_parts = pd.read_excel(book_path, sheet_name="HZ")
vt_parts = pd.read_excel(book_path, sheet_name="VT")

stats = pd.read_excel(book_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")

data = stats.merge(daily, left_on="Part", right_on="Material")

# ==========================================
# PRODUCTION RATE
# ==========================================

data["Rate"] = (3600 / data["Cycle time "].replace(0, np.nan)) * data["cavity"]
data = data[data["Rate"].notna()]

# ==========================================
# PARTS AND MACHINES
# ==========================================

parts = data["Material"].unique().tolist()
machines = hz_matrix.columns[1:].tolist()

# ==========================================
# LOOKUP DICTIONARIES
# ==========================================

inventory = dict(zip(data["Material"], data["Inventory on 24th"]))
demand = dict(zip(data["Material"], data["2026-03-12 Total Production Plan"]))
rate = dict(zip(data["Material"], data["Rate"]))

# replace NaN
inventory = {k: (0 if pd.isna(v) else v) for k, v in inventory.items()}
demand = {k: (0 if pd.isna(v) else v) for k, v in demand.items()}

# ==========================================
# COMPATIBILITY MATRIX
# ==========================================

compatibility = {}

for _, row in hz_matrix.iterrows():
    part = row["Part"]
    for m in machines:
        compatibility[(part, m)] = row[m]

# ==========================================
# CREATE DECISION VARIABLES
# ==========================================

variables = []
var_index = {}

k = 0

for p in parts:
    for m in machines:

        if compatibility.get((p, m), 0) == 1:

            variables.append((p, m))
            var_index[(p, m)] = k
            k += 1

# ==========================================
# SHORTAGE VARIABLES
# ==========================================

shortage_index = {}

for p in parts:

    variables.append(("shortage", p))
    shortage_index[p] = len(variables) - 1

n = len(variables)

# ==========================================
# OBJECTIVE FUNCTION
# ==========================================

c = np.zeros(n)

for p in parts:
    idx = shortage_index[p]
    c[idx] = SHORTAGE_PENALTY

# ==========================================
# CONSTRAINT MATRICES
# ==========================================

A = []
b = []

# ------------------------------------------
# MACHINE CAPACITY
# ------------------------------------------

for m in machines:

    row = np.zeros(n)

    for p in parts:
        if (p, m) in var_index:
            idx = var_index[(p, m)]
            row[idx] = 1 / rate[p]

    A.append(row)
    b.append(AVAILABLE_HOURS)

# ------------------------------------------
# DEMAND SATISFACTION
# ------------------------------------------

for p in parts:

    row = np.zeros(n)

    for m in machines:
        if (p, m) in var_index:
            idx = var_index[(p, m)]
            row[idx] = -1

    row[shortage_index[p]] = -1

    A.append(row)
    b.append(inventory[p] - demand[p])

# ==========================================
# SOLVE OPTIMIZATION
# ==========================================

result = linprog(
    c,
    A_ub=np.array(A),
    b_ub=np.array(b),
    bounds=(0, None),
    method="highs"
)

print("Solver success:", result.success)
print("Solver message:", result.message)

if not result.success:
    raise Exception("Optimization failed")

solution = result.x

# ==========================================
# EXTRACT PLAN
# ==========================================

plan = []

for (p, m), idx in var_index.items():

    qty = solution[idx]

    if qty > 1:

        plan.append({
            "Part": p,
            "Machine": m,
            "Qty": round(qty, 0),
            "Run_Hours": round(qty / rate[p], 2)
        })

plan_df = pd.DataFrame(plan)

plan_df.to_excel("Smart_APS_Plan_V1.xlsx", index=False)

print("Optimization Completed")

In [ ]:
import pandas as pd
import numpy as np

# =================================================
# PARAMETERS
# =================================================

AVAILABLE_HOURS = 22
MIN_RUN_HOURS = 4

book_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_13march_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

# =================================================
# LOAD DATA
# =================================================

stats = pd.read_excel(book_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")
vt_matrix = pd.read_excel(matrix_path, sheet_name="VT_Matrix")

data = stats.merge(daily, left_on="Part", right_on="Material")

# =================================================
# PRODUCTION RATE
# =================================================

data["Rate"] = (3600 / data["Cycle time "].replace(0, np.nan)) * data["cavity"]
data = data[data["Rate"].notna()]

# =================================================
# LOOKUP DICTIONARIES
# =================================================

inventory = dict(zip(data["Material"], data["Inventory on 24th"]))
demand = dict(zip(data["Material"], data["2026-03-12 Total Production Plan"]))
rate = dict(zip(data["Material"], data["Rate"]))

inventory = {k: (0 if pd.isna(v) else v) for k, v in inventory.items()}
demand = {k: (0 if pd.isna(v) else v) for k, v in demand.items()}

# =================================================
# BUILD COMPATIBILITY
# =================================================

def build_compatibility(matrix):

    machines = matrix.columns[1:].tolist()
    compatibility = {}

    for _, row in matrix.iterrows():

        part = row["Part"]

        for m in machines:

            if row[m] == 1:

                compatibility.setdefault(part, []).append(m)

    return compatibility, machines


hz_compat, hz_machines = build_compatibility(hz_matrix)
vt_compat, vt_machines = build_compatibility(vt_matrix)

# =================================================
# PRIORITY ENGINE
# =================================================

def compute_priority(parts):

    rows = []

    for p in parts:

        inv = inventory.get(p, 0)
        dem = demand.get(p, 0)
        r = rate.get(p, 1)

        shortage = max(0, dem - inv)

        pressure = shortage / r

        rows.append({
            "Part": p,
            "Shortage": shortage,
            "Pressure": pressure
        })

    df = pd.DataFrame(rows)

    return df.sort_values("Pressure", ascending=False)


# =================================================
# MACHINE SCHEDULER
# =================================================

def schedule(parts, compatibility, machines):

    machine_hours = {m: 0 for m in machines}

    plan = []
    not_planned = []

    priority_df = compute_priority(parts)

    for _, row in priority_df.iterrows():

        part = row["Part"]

        dem = demand.get(part, 0)
        inv = inventory.get(part, 0)

        required_qty = max(0, dem - inv)

        if required_qty == 0:

            continue

        r = rate[part]

        required_hours = required_qty / r

        assigned = False

        for m in compatibility.get(part, []):

            used = machine_hours[m]

            free = AVAILABLE_HOURS - used

            if free >= MIN_RUN_HOURS:

                run_hours = min(required_hours, free)

                if run_hours < MIN_RUN_HOURS:
                    run_hours = MIN_RUN_HOURS

                qty = run_hours * r

                machine_hours[m] += run_hours

                plan.append({
                    "Part": part,
                    "Machine": m,
                    "Run_Hours": round(run_hours,2),
                    "Production_Qty": round(qty,0)
                })

                assigned = True
                break

        if not assigned:

            not_planned.append({
                "Part": part,
                "Reason": "No machine capacity"
            })

    return pd.DataFrame(plan), pd.DataFrame(not_planned)


# =================================================
# RUN HZ AND VT
# =================================================

hz_parts = data[data["Material"].isin(hz_matrix["Part"])]["Material"].unique()
vt_parts = data[data["Material"].isin(vt_matrix["Part"])]["Material"].unique()

hz_plan, hz_not = schedule(hz_parts, hz_compat, hz_machines)
vt_plan, vt_not = schedule(vt_parts, vt_compat, vt_machines)

# =================================================
# SAVE OUTPUT
# =================================================

with pd.ExcelWriter("Smart_APS_Plan_V2.xlsx") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_not.to_excel(writer, sheet_name="HZ_Not_Planned", index=False)
    vt_not.to_excel(writer, sheet_name="VT_Not_Planned", index=False)

print("Smart APS Planning Completed")